<a href="https://colab.research.google.com/github/Harshitshukla0/notebooks/blob/main/VoC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


#!/usr/bin/env python3
"""
poc_daily_prioritizer.py

Single-file POC pipeline that:
- loads Medallia-like feedback data (simulated or from CSV)
- loads customer LPV data (simulated or from CSV)
- assigns each feedback to one of 5 SMEs (configurable charters)
- computes a priority score combining an LLM-based score and LPV
- selects top-N items per SME and creates a daily email digest
- optionally sends emails via SMTP (use with care — default is simulation)

Usage:
    python poc_daily_prioritizer.py            # run with simulated data, print email digests
    python poc_daily_prioritizer.py --medal path/to/medallia.csv --cust path/to/customers.csv
    python poc_daily_prioritizer.py --send-email  # attempts to send via SMTP; configure env vars

Environment variables for SMTP sending (if using --send-email):
    SMTP_HOST, SMTP_PORT, SMTP_USER, SMTP_PASS, SMTP_FROM

Notes:
- Replace mocked_llm_score with a real LLM call adapter to call OpenAI/Anthropic/etc.
- Replace medallia/cust CSV loaders with API/BigQuery connectors in production.
- Keep SME charter config in YAML/DB for production; the POC uses in-script dict.
"""

import os
import argparse
from datetime import datetime, timedelta
import random
import math
import smtplib
from email.message import EmailMessage
import pandas as pd
import csv
import sys
from typing import Dict, Any

# ----------------------
# Config (tuneable)
# ----------------------
TOP_N = 10
SINCE_HOURS = 24
LLM_WEIGHT = 0.6
LPV_WEIGHT = 0.4
AUDIT_PATH = "./daily_selection_audit.csv"
SIMULATE_DATA = True  # set to False when providing real CSVs with --medal/--cust

# SME charter: list of keywords matched against product_area or topic
SMES = {
    "SME_A": {"name": "Alice", "email": "alice@example.com", "charter": ["billing", "payments"]},
    "SME_B": {"name": "Bob", "email": "bob@example.com", "charter": ["mobile", "performance"]},
    "SME_C": {"name": "Carol", "email": "carol@example.com", "charter": ["security", "auth"]},
    "SME_D": {"name": "Dan", "email": "dan@example.com", "charter": ["reports", "dashboard", "ux"]},
    "SME_E": {"name": "Eve", "email": "eve@example.com", "charter": ["file_service", "enterprise", "feature_request", "support"]},
}

# ----------------------
# Utility & Mock Data
# ----------------------
def create_sample_medallia(n=150, path: str = None):
    """Create a sample Medallia CSV for local testing and optionally save to path."""
    sample_verbatims = [
        ("The checkout crashes when I try to pay with card X", 1, "checkout", "payments"),
        ("App performance is slow on Android 11", 2, "performance", "mobile"),
        ("I love the new dashboard layout", 5, "ui", "dashboard"),
        ("Billing shows incorrect amount for subscription plan", 1, "billing", "billing"),
        ("Feature Y is missing essential options for enterprise", 2, "feature_request", "enterprise"),
        ("Unable to reset password via email flow", 1, "auth", "security"),
        ("Support response time was great", 5, "support", "support"),
        ("Confusing labels in reporting causing wrong interpretation", 3, "ux", "reports"),
        ("Crash when uploading large file (>500MB)", 1, "upload", "file_service"),
        ("Suggestion: allow CSV export of metrics", 4, "feature_request", "reports"),
    ]
    rows = []
    now = datetime.now()
    for idx in range(1, n + 1):
        base = random.choice(sample_verbatims)
        verb = base[0]
        rating = base[1]
        topic = base[2]
        area = base[3]
        # occasional urgent variations
        if random.random() < 0.15:
            verb = verb + " -- urgent"
            rating = min(2, rating)
        if random.random() < 0.1:
            verb = verb + " please fix"
        ts = now - timedelta(hours=random.randint(0, 48), minutes=random.randint(0, 59))
        feedback_id = f"FB{1000 + idx}"
        rows.append({
            "feedback_id": feedback_id,
            "verbatim": verb,
            "rating": rating,
            "topic": topic,
            "product_area": area,
            "created_at": ts.isoformat()
        })
    df = pd.DataFrame(rows)
    if path:
        df.to_csv(path, index=False)
    return df

def create_sample_customers(n=200, path: str = None):
    """Create a sample customer CSV with LPV and optionally save to path."""
    rows = []
    for i in range(1, n + 1):
        cust_id = f"CUST{i:04d}"
        lpv = max(0, int(random.gauss(5000, 3000)))
        tier = random.choice(["enterprise", "midmarket", "self-serve"])
        rows.append({"customer_id": cust_id, "lpv_1y": lpv, "tier": tier})
    df = pd.DataFrame(rows)
    if path:
        df.to_csv(path, index=False)
    return df

# ----------------------
# Connectors (POC vs production)
# ----------------------
def medallia_fetch_csv(path: str = None, since_hours: int = SINCE_HOURS) -> pd.DataFrame:
    """
    Load Medallia-like CSV.
    Expected columns: feedback_id, verbatim, rating, topic, product_area, created_at
    """
    if path:
        df = pd.read_csv(path, parse_dates=["created_at"])
    else:
        df = create_sample_medallia()
    cutoff = datetime.now() - timedelta(hours=since_hours)
    # some test data may use strings; ensure proper datetime
    if df["created_at"].dtype == "O":
        df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
    recent = df[df["created_at"] >= cutoff].copy()
    return recent

def customers_fetch_csv(path: str = None) -> pd.DataFrame:
    """
    Load customer LPV CSV with columns: customer_id, lpv_1y
    """
    if path:
        df = pd.read_csv(path)
    else:
        df = create_sample_customers()
    return df

# ----------------------
# Business logic
# ----------------------
def assign_sme(row: pd.Series, smes_cfg: Dict[str, Any]) -> str:
    """
    Assign an SME key based on product_area/topic keywords from the SME charter.
    Deterministic mapping: first matching SME in order of SMES dict.
    Fallback: urgent/low rating -> SME_E, else round-robin via hash.
    """
    area = str(row.get("product_area", "")).lower()
    topic = str(row.get("topic", "")).lower()
    for sme_key, cfg in smes_cfg.items():
        for kw in cfg.get("charter", []):
            if kw.lower() in area or kw.lower() in topic:
                return sme_key
    # fallback by severity
    rating = row.get("rating", 5)
    if pd.isna(rating):
        rating = 5
    try:
        rating_val = float(rating)
    except Exception:
        rating_val = 5
    if rating_val <= 2:
        return "SME_E"
    # deterministic fallback via hash of feedback_id
    fid = str(row.get("feedback_id", ""))
    idx = sum(ord(c) for c in fid) if fid else random.randint(0, 1000)
    keys = list(smes_cfg.keys())
    return keys[idx % len(keys)]

def mocked_llm_score(verbatim: str, rating: float) -> float:
    """
    Mocked LLM scoring: heuristics converting text + rating into a 0..1 float.
    Replace this function with a real LLM call adapter in production.
    """
    text = (verbatim or "").lower()
    score = 0.0
    urgent_keywords = ["crash", "unable", "incorrect", "urgent", "please fix", "fails", "error", "cannot", "can't", "fail"]
    for kw in urgent_keywords:
        if kw in text:
            score += 0.25
    # lower rating increases score
    try:
        score += max(0, (5 - float(rating))) * 0.05
    except Exception:
        score += 0.1
    # text length gives a small boost
    word_count = len(text.split())
    score += min(word_count / 100.0, 0.2)
    return min(max(score, 0.0), 1.0)

def batch_llm_scores(df: pd.DataFrame, text_col: str = "verbatim", rating_col: str = "rating") -> pd.Series:
    """Return a pandas Series of mocked LLM scores for each row."""
    return df.apply(lambda r: mocked_llm_score(r.get(text_col, ""), r.get(rating_col, 5)), axis=1)

def normalize_series(s: pd.Series) -> pd.Series:
    """Min-max normalize; handle constant series by returning 0.5 values."""
    if s.empty:
        return s
    mn = s.min()
    mx = s.max()
    if mx == mn:
        return pd.Series([0.5] * len(s), index=s.index)
    return (s - mn) / (mx - mn)

def prioritize_for_sme(df_sme: pd.DataFrame, customers_df: pd.DataFrame, top_n: int = TOP_N,
                       w_llm: float = LLM_WEIGHT, w_lpv: float = LPV_WEIGHT) -> pd.DataFrame:
    """
    For df_sme, synthesize or lookup customer_id -> lpv_1y, compute llm_score and lpv_norm,
    then return top_n rows sorted by priority_score.
    NOTE: in many real datasets there is a customer_id column to join on.
    """
    if df_sme.empty:
        return df_sme.copy()
    df = df_sme.copy().reset_index(drop=True)
    # synthesize a customer_id if none present (POC). In prod, join on real customer_id.
    if "customer_id" not in df.columns:
        # create a deterministic mapping to existing customers to keep LPV meaningful
        cust_list = customers_df.get("customer_id").tolist()
        if not cust_list:
            df["customer_id"] = None
            df["lpv_1y"] = customers_df.get("lpv_1y").median() if not customers_df.empty else 0
        else:
            df["customer_id"] = df.index.map(lambda i: cust_list[i % len(cust_list)])
    # map LPV
    cust_map = customers_df.set_index("customer_id")["lpv_1y"].to_dict() if not customers_df.empty else {}
    df["lpv_1y"] = df["customer_id"].map(cust_map).fillna(customers_df["lpv_1y"].median() if not customers_df.empty else 0)
    # llm score (mocked) - in prod, replace with batch LLM call adapter
    df["llm_score"] = batch_llm_scores(df)
    df["llm_norm"] = normalize_series(df["llm_score"])
    df["lpv_norm"] = normalize_series(df["lpv_1y"])
    # composite
    df["priority_score"] = df["llm_norm"] * w_llm + df["lpv_norm"] * w_lpv
    # stable tie-breakers: rating low -> higher, created_at older -> lower priority normally, so we prefer newer
    df = df.sort_values(["priority_score", "rating", "created_at"], ascending=[False, True, False])
    return df.head(top_n)

# ----------------------
# Email formatting & sending
# ----------------------
def format_email_for_sme(sme_key: str, selection_df: pd.DataFrame) -> Dict[str, str]:
    cfg = SMES[sme_key]
    subject = f"[DAILY P1] Top {len(selection_df)} feedback items for {cfg['name']} - {datetime.now().date().isoformat()}"
    lines = []
    lines.append(f"Hi {cfg['name']},")
    lines.append("")
    lines.append("Below are the top feedback items assigned to you based on charter + automated prioritization.")
    lines.append("Copy the feedback_id to begin your investigation and follow your normal process.")
    lines.append("")
    for _, row in selection_df.iterrows():
        fid = row.get("feedback_id", "")
        score = row.get("priority_score", 0)
        rating = row.get("rating", "")
        lpv = int(row.get("lpv_1y", 0))
        verb = (row.get("verbatim", "") or "")[:280].replace("\n", " ")
        lines.append(f"- {fid} | score:{score:.3f} | rating:{rating} | lpv:{lpv} | {verb}")
    if selection_df.empty:
        lines.append("(No items in the last 24 hours matching your charter.)")
    lines.append("")
    lines.append("Regards,\nProduct Ops (POC)")
    body = "\n".join(lines)
    return {"to": cfg["email"], "cc": "manager@example.com", "subject": subject, "body": body}

def send_email_simulated(to: str, cc: str, subject: str, body: str):
    """Simulation of sending: print to console. Use for POC & demoing."""
    sep = "=" * 80
    print(sep)
    print(f"To: {to}  CC: {cc}")
    print(f"Subject: {subject}\n")
    print(body)
    print(sep)
    print()

def send_email_smtp(to: str, cc: str, subject: str, body: str):
    """
    Send email via SMTP using environment variables:
      SMTP_HOST, SMTP_PORT, SMTP_USER, SMTP_PASS, SMTP_FROM
    * Use only in staging / with permission.
    """
    smtp_host = os.environ.get("SMTP_HOST")
    smtp_port = int(os.environ.get("SMTP_PORT", "587"))
    smtp_user = os.environ.get("SMTP_USER")
    smtp_pass = os.environ.get("SMTP_PASS")
    smtp_from = os.environ.get("SMTP_FROM", smtp_user)
    if not smtp_host or not smtp_user or not smtp_pass:
        raise RuntimeError("SMTP env variables not set: SMTP_HOST/SMTP_USER/SMTP_PASS required for sending.")
    msg = EmailMessage()
    msg["Subject"] = subject
    msg["From"] = smtp_from
    msg["To"] = to
    msg["Cc"] = cc
    msg.set_content(body)
    # envelope recipients
    recipients = [to] + ([cc] if cc else [])
    with smtplib.SMTP(smtp_host, smtp_port) as s:
        s.starttls()
        s.login(smtp_user, smtp_pass)
        s.send_message(msg, from_addr=smtp_from, to_addrs=recipients)

# ----------------------
# Audit & persistence
# ----------------------
def append_audit(audit_path: str, rows: list):
    """Append audit rows to a CSV. Create file with header if not exists."""
    if not rows:
        return
    fieldnames = list(rows[0].keys())
    write_header = not os.path.exists(audit_path)
    with open(audit_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        for r in rows:
            writer.writerow(r)

# ----------------------
# Orchestration
# ----------------------
def run_pipeline(medal_path: str = None, cust_path: str = None, send_email: bool = False):
    # 1) Ingest
    med_df = medallia_fetch_csv(path=medal_path, since_hours=SINCE_HOURS)
    customers_df = customers_fetch_csv(path=cust_path)

    # 2) Assign SMEs
    med_df["sme_key"] = med_df.apply(lambda r: assign_sme(r, SMES), axis=1)

    # 3) For each SME, prioritize and send digest
    audit_rows = []
    for sme_key in SMES.keys():
        df_sme = med_df[med_df["sme_key"] == sme_key]
        prioritized = prioritize_for_sme(df_sme, customers_df, top_n=TOP_N, w_llm=LLM_WEIGHT, w_lpv=LPV_WEIGHT)
        email_data = format_email_for_sme(sme_key, prioritized)
        # send (or simulate)
        try:
            if send_email:
                send_email_smtp(email_data["to"], email_data["cc"], email_data["subject"], email_data["body"])
            else:
                send_email_simulated(email_data["to"], email_data["cc"], email_data["subject"], email_data["body"])
        except Exception as e:
            print(f"Error sending email for {sme_key}: {e}", file=sys.stderr)
        # audit entries
        for _, r in prioritized.iterrows():
            audit_rows.append({
                "date": datetime.now().isoformat(),
                "sme": sme_key,
                "feedback_id": r.get("feedback_id"),
                "priority_score": float(r.get("priority_score", 0)),
                "rating": r.get("rating"),
                "lpv_1y": int(r.get("lpv_1y", 0))
            })
    # 4) persist audit
    append_audit(AUDIT_PATH, audit_rows)
    print(f"Pipeline run complete. Audit appended to: {AUDIT_PATH}")

# ----------------------
# CLI
# ----------------------
def parse_args():
    p = argparse.ArgumentParser(description="POC: daily prioritizer pipeline")
    p.add_argument("--medal", help="Path to Medallia CSV (if omitted, simulated data used)")
    p.add_argument("--cust", help="Path to customer CSV (if omitted, simulated data used)")
    p.add_argument("--send-email", action="store_true", help="Actually send emails using SMTP env vars (dangerous in prod)")
    p.add_argument("--top-n", type=int, default=TOP_N, help="Top N items per SME")
    return p.parse_args()

def main():
    args = parse_args()
    global TOP_N
    TOP_N = args.top_n
    run_pipeline(medal_path=args.medal, cust_path=args.cust, send_email=args.send_email)

if __name__ == "__main__":
    main()
